# ML Pipelines

MLlib standardizes APIs for machine learning algorithms to make it easier to combine multiple algorithms into a single pipeline, or workflow. 

This section covers the key concepts introduced by the Pipelines API, where the pipeline concept is mostly inspired by the scikit-learn project.

## Setup Spark

In [1]:
import findspark
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Zoo").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 10:15:49 WARN Utils: Your hostname, NicSBook, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/23 10:15:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 10:15:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark

## DataFrame

Machine learning can be applied to a wide variety of data types, such as vectors, text, images, and structured data. This API adopts the DataFrame from Spark SQL in order to support a variety of data types.

DataFrame supports many basic and structured types; see the Spark SQL datatype reference for a list of supported types. In addition to the types listed in the Spark SQL guide, DataFrame can use ML Vector types.

A DataFrame can be created either implicitly or explicitly from a regular RDD. See the code examples below and the Spark SQL programming guide for examples.

Columns in a DataFrame are named. The code examples below use names such as “text”, “features”, and “label”.

## Pipeline components

### Transformers
A Transformer is an abstraction that includes feature transformers and learned models. Technically, a Transformer implements a method transform(), which converts one DataFrame into another, generally by appending one or more columns. For example:

A feature transformer might take a DataFrame, read a column (e.g., text), map it into a new column (e.g., feature vectors), and output a new DataFrame with the mapped column appended.
A learning model might take a DataFrame, read the column containing feature vectors, predict the label for each feature vector, and output a new DataFrame with predicted labels appended as a column.

### Estimators
An Estimator abstracts the concept of a learning algorithm or any algorithm that fits or trains on data. Technically, an Estimator implements a method fit(), which accepts a DataFrame and produces a Model, which is a Transformer. For example, a learning algorithm such as LogisticRegression is an Estimator, and calling fit() trains a LogisticRegressionModel, which is a Model and hence a Transformer.

### Pipeline
In machine learning, it is common to run a sequence of algorithms to process and learn from data. E.g., a simple text document processing workflow might include several stages:

Split each document’s text into words.
Convert each document’s words into a numerical feature vector.
Learn a prediction model using the feature vectors and labels.
MLlib represents such a workflow as a Pipeline, which consists of a sequence of PipelineStages (Transformers and Estimators) to be run in a specific order. We will use this simple workflow as a running example in this section.

### How it works
A Pipeline is specified as a sequence of stages, and each stage is either a Transformer or an Estimator. These stages are run in order, and the input DataFrame is transformed as it passes through each stage. For Transformer stages, the transform() method is called on the DataFrame. For Estimator stages, the fit() method is called to produce a Transformer (which becomes part of the PipelineModel, or fitted Pipeline), and that Transformer’s transform() method is called on the DataFrame.

We illustrate this for the simple text document workflow. The figure below is for the training time usage of a Pipeline.
![](https://spark.apache.org/docs/latest/img/ml-Pipeline.png)


Above, the top row represents a Pipeline with three stages. The first two (Tokenizer and HashingTF) are Transformers (blue), and the third (LogisticRegression) is an Estimator (red). The bottom row represents data flowing through the pipeline, where cylinders indicate DataFrames. The Pipeline.fit() method is called on the original DataFrame, which has raw text documents and labels. The Tokenizer.transform() method splits the raw text documents into words, adding a new column with words to the DataFrame. The HashingTF.transform() method converts the words column into feature vectors, adding a new column with those vectors to the DataFrame. Now, since LogisticRegression is an Estimator, the Pipeline first calls LogisticRegression.fit() to produce a LogisticRegressionModel. If the Pipeline had more Estimators, it would call the LogisticRegressionModel’s transform() method on the DataFrame before passing the DataFrame to the next stage.

A Pipeline is an Estimator. Thus, after a Pipeline’s fit() method runs, it produces a PipelineModel, which is a Transformer. This PipelineModel is used at test time; the figure below illustrates this usage.

![](https://spark.apache.org/docs/latest/img/ml-PipelineModel.png)

In the figure above, the PipelineModel has the same number of stages as the original Pipeline, but all Estimators in the original Pipeline have become Transformers. When the PipelineModel’s transform() method is called on a test dataset, the data are passed through the fitted pipeline in order. Each stage’s transform() method updates the dataset and passes it to the next stage.

Pipelines and PipelineModels help to ensure that training and test data go through identical feature processing steps.

## ML persistence: Saving and Loading Pipelines
Often times it is worth it to save a model or a pipeline to disk for later use. In Spark 1.6, a model import/export functionality was added to the Pipeline API. As of Spark 2.3, the DataFrame-based API in spark.ml and pyspark.ml has complete coverage.

ML persistence works across Scala, Java and Python. 

## Examples

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import HashingTF, Tokenizer

# Prepare training documents from a list of (id, text, label) tuples.
training = spark.createDataFrame([
    (0, "a b c d e spark", 1.0),
    (1, "b d", 0.0),
    (2, "spark f g h", 1.0),
    (3, "hadoop mapreduce", 0.0)
], ["id", "text", "label"])
training.show()

+---+----------------+-----+
| id|            text|label|
+---+----------------+-----+
|  0| a b c d e spark|  1.0|
|  1|             b d|  0.0|
|  2|     spark f g h|  1.0|
|  3|hadoop mapreduce|  0.0|
+---+----------------+-----+



In [4]:
# Configure an ML pipeline, which consists of three stages: tokenizer, hashingTF, and lr.
tokenizer = Tokenizer(inputCol="text", outputCol="words")
hashingTF = HashingTF(inputCol=tokenizer.getOutputCol(), outputCol="features")
lr = LogisticRegression(maxIter=10, regParam=0.001)
pipeline = Pipeline(stages=[tokenizer, hashingTF, lr])

In [9]:
tokenizer.transform(training).show(truncate=False)

+---+----------------+-----+----------------------+
|id |text            |label|words                 |
+---+----------------+-----+----------------------+
|0  |a b c d e spark |1.0  |[a, b, c, d, e, spark]|
|1  |b d             |0.0  |[b, d]                |
|2  |spark f g h     |1.0  |[spark, f, g, h]      |
|3  |hadoop mapreduce|0.0  |[hadoop, mapreduce]   |
+---+----------------+-----+----------------------+



In [10]:
# Fit the pipeline to training documents.
model = pipeline.fit(training)
model

PipelineModel_b0b81e44ddff

In [11]:
# Prepare test documents, which are unlabeled (id, text) tuples.
test = spark.createDataFrame([
    (4, "a b c d e spark"),
    (5, "batman"),
    (6, "spark a hadoop"),
    (7, "apache hadoop")
], ["id", "text"])
test.show()

+---+---------------+
| id|           text|
+---+---------------+
|  4|a b c d e spark|
|  5|         batman|
|  6| spark a hadoop|
|  7|  apache hadoop|
+---+---------------+



In [12]:
# Make predictions on test documents and print columns of interest.
prediction = model.transform(test)

In [13]:
prediction.printSchema()

root
 |-- id: long (nullable = true)
 |-- text: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- features: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)



In [14]:
prediction.select('text','prediction').show()

+---------------+----------+
|           text|prediction|
+---------------+----------+
|a b c d e spark|       1.0|
|         batman|       0.0|
| spark a hadoop|       1.0|
|  apache hadoop|       0.0|
+---------------+----------+



In [15]:
selected = prediction.select("id", "text", "probability", "prediction")
for row in selected.collect():
    rid, text, prob, prediction = row
    print("(%d, %s) --> prob=%s, prediction=%f" % (rid, text, str(prob), prediction))

(4, a b c d e spark) --> prob=[0.0026282134969420217,0.997371786503058], prediction=1.000000
(5, batman) --> prob=[0.9847700067623042,0.015229993237695805], prediction=0.000000
(6, spark a hadoop) --> prob=[0.29643349723932283,0.7035665027606772], prediction=1.000000
(7, apache hadoop) --> prob=[0.9955732114398529,0.00442678856014711], prediction=0.000000


In [16]:
# Make predictions on training documents and print columns of interest.
prediction = model.transform(training)
selected = prediction.select("id", "text", "probability", "prediction")
for row in selected.collect():
    rid, text, prob, prediction = row
    print("(%d, %s) --> prob=%s, prediction=%f" % (rid, text, str(prob), prediction))

(0, a b c d e spark) --> prob=[0.0026282134969420217,0.997371786503058], prediction=1.000000
(1, b d) --> prob=[0.9963902711801113,0.0036097288198887467], prediction=0.000000
(2, spark f g h) --> prob=[0.002208105057026994,0.997791894942973], prediction=1.000000
(3, hadoop mapreduce) --> prob=[0.9987232337063715,0.0012767662936284951], prediction=0.000000


In [17]:
model.stages

[Tokenizer_fa5511ddc011,
 HashingTF_1f887ba6acce,
 LogisticRegressionModel: uid=LogisticRegression_5424a96bead1, numClasses=2, numFeatures=262144]

In [18]:
# Extract 
# Extract the summary from the returned LogisticRegressionModel instance trained
# in the earlier example
modelSummary = model.stages[2].summary

modelSummary.accuracy

1.0

In [19]:
test2 = spark.createDataFrame([
    (8, "spark d f")
], ["id", "text"])

# Make predictions on training documents and print columns of interest.
prediction = model.transform(test2)
selected = prediction.select("id", "text", "probability", "prediction")
for row in selected.collect():
    rid, text, prob, prediction = row
    print("(%d, %s) --> prob=%s, prediction=%f" % (rid, text, str(prob), prediction))

(8, spark d f) --> prob=[0.2769645993535446,0.7230354006464554], prediction=1.000000


In [20]:
spark.stop()